In [ ]:
pip install pinecone sentence-transformers pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 14.6 MB/s eta 0:00:00


# PDF Embedding + Pinecone Vector Database


In [ ]:
import pdfplumber
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import uuid

In [ ]:
PINECONE_API_KEY = "picecone"

pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
# STEP 2 : CREATE INDEX
index_name = "pdf-mcq-index"

# Create index only first time
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",

        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index Ready")

Index Ready


In [ ]:
# Connect index
index = pc.Index(index_name)

In [ ]:
# STEP 3 : LOAD EMBEDDING MODEL
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# STEP 4 : EXTRACT TEXT FROM PDF


def extract_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


In [ ]:
# STEP 5 : TEXT CHUNKING
def chunk_text(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        chunks.append(chunk)

    return chunks

In [ ]:
# STEP 6 : CREATE EMBEDDINGS

pdf_path = "/content/budget_speech.pdf"

text = extract_text(pdf_path)

chunks = chunk_text(text)

print("Total Chunks:", len(chunks))

# Create embeddings
embeddings = model.encode(chunks)

Total Chunks: 199


In [ ]:
# STEP 7 : STORE IN PINECONE


vectors = []

for i, chunk in enumerate(chunks):

    vector = {
        "id": str(uuid.uuid4()),

        "values": embeddings[i].tolist(),

        "metadata": {
            "text": chunk
        }
    }

    vectors.append(vector)


In [ ]:

# Upload to Pinecone
index.upsert(vectors=vectors)

print("Embeddings stored in Pinecone")

Embeddings stored in Pinecone


In [ ]:
'''

#not runn all


In [ ]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# --------------------------------
# CONNECT PINECONE
# --------------------------------

pc = Pinecone(
    PINECONE_API_KEY = "pinecone"
)

index = pc.Index("pdf-mcq-index")

# --------------------------------
# LOAD EMBEDDING MODEL
# --------------------------------

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
query = "What is the name of the dialogue where innovative ideas were shared with the Prime Minister?"

# Convert query to embedding
query_embedding = model.encode(query).tolist()

# Search in Pinecone
results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

# Show results
for match in results['matches']:

    print(f"\n\nScore:, {match['score']}\n")

    print(match['metadata']['text'])



Score:, 0.40167141

5. As I begin Part A, I want to express my gratitude to the people for
standing firmly with us as we forge our way together towards becoming
one of the largest economies of the world.
6. Our aim is to transform aspiration into achievement and potential
into performance, as we ensure that the dividends of growth reach every
farmer, the scheduled caste, the scheduled tribes, the nomads, the youth,
the poor and the women.
7. In the Viksit Bharat Young Leaders Dialogue 2026, several
innovative ideas


Score:, 0.389954567

hoices we have made,
even in times of heightened uncertainty and disruption. Our
Government, led by Hon’ble Prime Minister Modi, has decisively and
consistently chosen action over ambivalence, reform over rhetoric and
people over populism.
2. We have pursued far reaching structural reforms,
fiscal prudence and monetary stability whilst
maintaining a strong thrust on public investment.
Keeping atmanirbharta as a lodestar, we have built domestic
manufa

In [ ]:
#!pip install groq pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.8 MB/s eta 0:00:00


In [ ]:
prompt = f"""
Generate 3 MCQ questions from this text.

 Rules-
    - Not provid answer
Text:
{context}
"""

In [ ]:
from groq import Groq

# Create client
client = Groq(
    api_key='groq'
)

In [ ]:
response = client.chat.completions.create(

    model="llama-3.3-70b-versatile",

    messages=[

        {
            "role": "system",
            "content": "You are an expert MCQ generator."
        },

        {
            "role": "user",
            "content": prompt
        }

    ],

    temperature=0.5,

    max_tokens=1000
)

In [ ]:
print(response.choices[0].message.content)

Here are 3 MCQ questions based on the text:

1. What is the author's goal for the country as mentioned in the text?
A) To become a developed nation by 2025
B) To transform aspiration into achievement and potential into performance
C) To reduce the population of the country
D) To increase the number of nomads in the country

2. According to the text, who are the beneficiaries of the country's growth?
A) Only the scheduled caste and scheduled tribes
B) Only the farmers and the youth
C) Every farmer, the scheduled caste, the scheduled tribes, the nomads, the youth, the poor, and the women
D) Only the rich and the wealthy

3. What event is mentioned in the text where several innovative ideas were discussed?
A) Viksit Bharat Economic Summit
B) Viksit Bharat Young Leaders Dialogue 2026
C) Bharat Youth Conference
D) Global Leaders Meet 2025
